## 1. Data Preparation From Multiple Files into a Single CSV

The dataset contains 36733 instances of 11 sensor measures aggregated over one hour, from a gas turbine located in Turkey for the purpose of studying flue gas emissions, namely CO and NOx.

- The dataset is stored in multiple files in S3 bucket. We will read all the files and combine them into a single CSV file for further processing.

- Reference: 
    - https://archive.ics.uci.edu/dataset/551/gas+turbine+co+and+nox+emission+data+set
    - https://www.timeanddate.com/holidays/turkey/


In [1]:
# If you're using the default bucket, set DEFAULT_BUCKET = True; 
# otherwise, if you're using a specific bucket, set it to False instead
DEFAULT_BUCKET = False

In [15]:
# import libraries
import boto3
import sagemaker
import pandas as pd
import os
import tqdm
import numpy as np

# Initialize Sagemaker account
s3 = boto3.resource("s3")
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")
# Sagemaker sessions
# High-level sagemaker session
sess = sagemaker.Session()
# Low-level client sagemaker and s3 clients
sm = boto3.Session().client(service_name="sagemaker", region_name=region)
s3 = boto3.client('s3')

# Bucket selection
if DEFAULT_BUCKET is True:
    # Code to read/write using the default bucket
    bucket = sess.default_bucket()
else:
    # Code to use a previously existing bucket
    bucket = "usdmsaai540-spring2026-team1"

paginator = s3.get_paginator('list_objects_v2')
page_iterator = paginator.paginate(Bucket=bucket, Prefix=prefix)

# Set Data Paths

In [6]:
public_path_csv= "../Data"

In [7]:
%store public_path_csv

Stored 'public_path_csv' (str)


In [8]:
bucket

'usdmsaai540-spring2026-team1'

In [10]:
# Private S3 Dataset path
s3_private_path_csv = "s3://usdmsaai540-spring2026-team1/CCPP/data/uci_dataset/"
print(s3_private_path_csv)

s3://usdmsaai540-spring2026-team1/CCPP/data/uci_dataset/


In [11]:
%store s3_private_path_csv

Stored 's3_private_path_csv' (str)


In [12]:
# if we're using the default bucket, we assume the data must be uploaded into it
if DEFAULT_BUCKET is True:
    !aws s3 cp --recursive $public_path_csv/ $s3_private_path_csv/ --exclude "*" --include "*.tsv"
    !aws s3 cp --recursive $public_path_csv/ $s3_private_path_csv/ --exclude "*" --include "*.csv"

**Checking all the files in the S3 folder:**

In [22]:
prefix = 'CCPP/data/uci_dataset/'
file_names = []

for page in page_iterator:
    if 'Contents' in page:
        for obj in page['Contents']:
            key = obj['Key']
            # Skip the folder itself
            if key == prefix:
                continue
            # Extract the file name from the key
            file_name = obj['Key'][len(prefix):]
            # Add the file name to the list
            file_names.append(file_name)

# Now file_names contains only the names of the files
print(file_names)

['CCPP 2011-2015 Full Weather Data.csv', 'TurkishHolidays.tsv', 'gt_2011_wDate.csv', 'gt_2012_wDate.csv', 'gt_2013_wDate.csv', 'gt_2014_wDate.csv', 'gt_2015_wDate.csv']


In [24]:
# Confirm number of files
print(len(file_names))

7


In [25]:
# Load gt_2011–2015 CSV files from private S3 and store as a dataframe list
gt=[]
for i in range(1,6):
    df = pd.read_csv(f"{s3_private_path_csv}gt_201{i}_wDate.csv")
    gt.append(df)
# Stack all dataframes from gt into one dataframe
gt_all_years = pd.concat(gt,ignore_index=True)

In [27]:
gt_all_years.head()

,DT,AT,AP,AH,AFDP,GTEP,TIT,TAT,TEY,CDP,CO,NOX
0,2011-01-01T00:00:00,4.5878,1018.7,83.675,3.5758,23.979,1086.2,549.83,134.67,11.898,0.32663,81.952
1,2011-01-01T01:00:00,4.2932,1018.3,84.235,3.5709,23.951,1086.1,550.05,134.67,11.892,0.44784,82.377
2,2011-01-01T02:00:00,3.9045,1018.4,84.858,3.5828,23.990,1086.5,550.19,135.10,12.042,0.45144,83.776
3,2011-01-01T03:00:00,3.7436,1018.3,85.434,3.5808,23.911,1086.5,550.17,135.03,11.990,0.23107,82.505
4,2011-01-01T04:00:00,3.7516,1017.8,85.182,3.5781,23.917,1085.9,550.00,134.67,11.910,0.26747,82.028


In [12]:
# Show data
gt_all_years.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36733 entries, 0 to 36732
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   DT      36733 non-null  object 
 1   AT      36733 non-null  float64
 2   AP      36733 non-null  float64
 3   AH      36733 non-null  float64
 4   AFDP    36733 non-null  float64
 5   GTEP    36733 non-null  float64
 6   TIT     36733 non-null  float64
 7   TAT     36733 non-null  float64
 8   TEY     36733 non-null  float64
 9   CDP     36733 non-null  float64
 10  CO      36733 non-null  float64
 11  NOX     36733 non-null  float64
dtypes: float64(11), object(1)
memory usage: 3.4+ MB


## Store the data to a CSV file, upload to private S3

In [24]:
from io import StringIO
csv_buffer = StringIO()
gt_all_years.to_csv(csv_buffer,index=False)

s3.put_object(
    Bucket=bucket,
    Key="CCPP/data/processeddata/gas_turbine.csv",
    Body=csv_buffer.getvalue()
)

{'ResponseMetadata': {'RequestId': 'SC5MA2SEG8S4HCZJ',
  'HostId': 'vOyxWiV5uhaMz6Rs5wNUH+Zu4jLySM+S+igi4nh9oQhf4WnfQdb/rjKSEOuxElT6TMhFbw+ZXvg=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'vOyxWiV5uhaMz6Rs5wNUH+Zu4jLySM+S+igi4nh9oQhf4WnfQdb/rjKSEOuxElT6TMhFbw+ZXvg=',
   'x-amz-request-id': 'SC5MA2SEG8S4HCZJ',
   'date': 'Tue, 10 Feb 2026 02:07:15 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"bfa5c0c2526f243849e83a58f6fd5ca7"',
   'x-amz-checksum-crc32': '3MeP+A==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"bfa5c0c2526f243849e83a58f6fd5ca7"',
 'ChecksumCRC32': '3MeP+A==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

In [28]:
# Read merged file from private s3 into dataframe
df = pd.read_csv(f"s3://{bucket}/CCPP/data/processeddata/gas_turbine.csv")

In [1]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>